In [ ]:
import pickle

import geopandas as gpd
import osmnx as ox
import polars as pl
from pyproj import Transformer
from valhalla import Actor, get_config

transformer = Transformer.from_crs("EPSG:4326", "EPSG:3857", always_xy=True)

ox.settings.log_console = False
ox.settings.use_cache = True

In [2]:
# def read_street_edge_list(device_group: int, path: str) -> list:
#     print(f"{device_group:02x}")
#     try:
#         with open(path, "br") as fp:
#             street_edge_list = pickle.load(fp)
#     except FileNotFoundError:
#         print(f"{path} was not found, generating it...")
#         df = prepare_parquet(device_group, days=opts.days, dir=opts.dir)
#         # runs more than 15 min on a full partition
#         street_edge_list = ox.distance.nearest_edges(
#             g_, X=df.geometry.x, Y=df.geometry.y
#         )  # , interpolate=25)
#         with open(path, "bw") as fp:
#             pickle.dump(street_edge_list, fp)
#     return street_edge_list

In [2]:
config = get_config(
    tile_extract="../data/valhalla/hungary/valhalla_tiles.tar",
    tile_dir="../data/valhalla/hungary/",
)
actor = Actor(config)

In [3]:
DIR = "/mnt/common-ssd/group-dirs/tkom-2026-project/pickwell/budapest/partitioned/"

In [4]:
for i in range(1):
    df = pl.read_parquet(f"{DIR}/{i:02x}/pings.parquet")

In [6]:
# actor.trace_route()

In [7]:
# request = {
#     "shape": [
#         {"lat": 47.47992, "lon": 19.074155, "type": "break"},
#         {"lat": 47.479995, "lon": 19.074221, "type": "via"},
#         {"lat": 47.476752, "lon": 19.075154, "type": "break"},
#     ],
#     "costing": "auto",
#     # "shape_match": "edge_walk",
# }

# result = actor.trace_route(request)

In [8]:
# request = {
#     "shape": [
#         {"lat": 47.47992, "lon": 19.074155, "type": "break"},
#         {"lat": 47.479995, "lon": 19.074221, "type": "via"},
#         {"lat": 47.476752, "lon": 19.075154, "type": "break"},
#     ],
#     "costing": "auto",
#     # "shape_match": "edge_walk",
# }

# result = actor.trace_attributes(request)

In [5]:
area = gpd.read_file("../data/osm/budapest/boundary.geojson")

In [ ]:
# try:
#     g_drive = ox.load_graphml("../output/budapest/drive.graphml")
# except FileNotFoundError:
#     # 2m 1s
#     g_drive = ox.graph_from_polygon(
#         area.geometry[0], network_type="drive", simplify=False
#     )

#     ox.save_graphml(g_drive, "../output/budapest/drive.graphml")

In [7]:
try:
    g_all = ox.load_graphml("../output/budapest/all.graphml")
except FileNotFoundError:
    # 4m
    g_all = ox.graph_from_polygon(area.geometry[0], network_type="all", simplify=False)
    ox.save_graphml(g_all, "../output/budapest/all.graphml")

In [8]:
g_all_proj = ox.project_graph(g_all, to_crs=3857)

In [9]:
xs, ys = transformer.transform(
    df["longitude"].to_numpy(),
    df["latitude"].to_numpy(),
)

df_proj = df.with_columns(
    pl.Series("lon", xs),
    pl.Series("lat", ys),
)

In [16]:
try:
    with open("../output/budapest/street_edge_list.pickle", "rb") as fp:
        street_edge_list = pickle.load(fp)
except FileExistsError:
    street_edge_list = ox.distance.nearest_edges(
        g_all_proj,
        X=df_proj["lon"],
        Y=df_proj["lat"],
        return_dist=True,
    )
    # 7m 45s
    with open("../output/budapest/street_edge_list.pickle", "wb") as fp:
        pickle.dump(street_edge_list, fp)

In [17]:
len(street_edge_list[0])

15460994

In [19]:
len(df_proj)

15460994

In [20]:
df_proj

timestamp,device_id,device_aid_type,latitude,longitude,id_with_type,lon,lat
"datetime[μs, Europe/Budapest]",i64,str,f64,f64,str,f64,f64
2024-10-02 02:37:09 CEST,1,"""AAID""",47.47992,19.074155,"""0000051fc26276495c5a9ab1753a70…",2.1233e6,6.0208e6
2024-10-02 03:17:49 CEST,1,"""AAID""",47.479995,19.074221,"""0000051fc26276495c5a9ab1753a70…",2.1233e6,6.0208e6
2024-10-02 03:26:50 CEST,1,"""AAID""",47.476752,19.075154,"""0000051fc26276495c5a9ab1753a70…",2.1234e6,6.0202e6
2024-10-02 03:27:50 CEST,1,"""AAID""",47.47999,19.07423,"""0000051fc26276495c5a9ab1753a70…",2.1233e6,6.0208e6
2024-10-02 04:06:48 CEST,1,"""AAID""",47.476752,19.075154,"""0000051fc26276495c5a9ab1753a70…",2.1234e6,6.0202e6
…,…,…,…,…,…,…,…
2025-10-24 15:16:40 CEST,6586,"""AAID""",47.588495,19.008983,"""00ffd75c779d5c2755207d57f66202…",2.1161e6,6.0387e6
2025-10-24 15:16:55 CEST,6586,"""AAID""",47.58826,19.009709,"""00ffd75c779d5c2755207d57f66202…",2.1162e6,6.0386e6
2025-10-24 15:17:04 CEST,6586,"""AAID""",47.588517,19.009056,"""00ffd75c779d5c2755207d57f66202…",2.1161e6,6.0387e6


In [21]:
osmids = [g_all.edges[u, v, key]["osmid"] for u, v, key in street_edge_list[0]]

In [ ]:
osmids

[25702955,
 25702955,
 577189673,
 25702955,
 577189673,
 25702955,
 577189673,
 25702955,
 25702955,
 261373301,
 54724399,
 35775318,
 1220449594,
 35775318,
 663022305,
 459647990,
 474449785,
 346277125,
 1329683630,
 548422398,
 548422397,
 458338077,
 848995617,
 848995617,
 272986118,
 848995617,
 272986118,
 848995617,
 272986118,
 272986118,
 272986118,
 272986118,
 272986118,
 154141667,
 405163097,
 405163097,
 405163097,
 272986118,
 272986118,
 272986118,
 1133079652,
 26377845,
 272986118,
 272986118,
 26377845,
 848995617,
 848995617,
 1036300873,
 26136319,
 26136319,
 1133079652,
 26136319,
 756452686,
 1532681963,
 474449785,
 311254910,
 184015418,
 1147450677,
 489635601,
 10221485,
 73031467,
 663022308,
 7988817,
 165713723,
 608833576,
 395458141,
 26377845,
 848995617,
 848995617,
 848995617,
 848995617,
 848995617,
 848995617,
 554989548,
 25702955,
 25702955,
 25702955,
 1133079652,
 25702961,
 1133079652,
 1133079652,
 1133079652,
 577189673,
 25702955,
 2613

: 

In [ ]:
# df.with_columns(street=osmids)